# Linear Regression with One Variable — California Housing Dataset

**Objective:** Predict median housing prices using a single feature (`AveRooms` — average number of rooms per dwelling).

**Methods Implemented:**
1. Gradient Descent (from scratch)
2. Normal Equation (closed-form solution)
3. Scikit-learn's LinearRegression (for verification)

**Evaluation Metrics:** MSE, RMSE, R² Score

---
## Cell 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# For cleaner plots in Colab
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

---
## Cell 2 — Load the Dataset

In [ ]:
# Load California Housing dataset (built into sklearn — no download needed)
housing = fetch_california_housing()

# Convert to DataFrame for easy exploration
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # Target: median house value (in $100,000s)

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"\nTarget: MedHouseVal (Median House Value in $100,000s)")
df.head()

---
## Cell 3 — Explore the Dataset

In [ ]:
# Basic statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"\nTotal samples: {len(df)}")
print(f"Missing values: {df.isnull().sum().sum()}")

print("\n--- Feature we will use: AveRooms ---")
print(f"Mean:   {df['AveRooms'].mean():.2f}")
print(f"Median: {df['AveRooms'].median():.2f}")
print(f"Std:    {df['AveRooms'].std():.2f}")
print(f"Min:    {df['AveRooms'].min():.2f}")
print(f"Max:    {df['AveRooms'].max():.2f}")

print("\n--- Target: MedHouseVal ---")
print(f"Mean:   {df['MedHouseVal'].mean():.2f} ($100K)")
print(f"Median: {df['MedHouseVal'].median():.2f} ($100K)")
print(f"Range:  {df['MedHouseVal'].min():.2f} — {df['MedHouseVal'].max():.2f} ($100K)")

---
## Cell 4 — Visualize the Raw Data (Scatter Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: AveRooms vs MedHouseVal
axes[0].scatter(df['AveRooms'], df['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
axes[0].set_xlabel('Average Rooms per Dwelling')
axes[0].set_ylabel('Median House Value ($100K)')
axes[0].set_title('AveRooms vs House Value (Raw Data)')
axes[0].set_xlim(0, 15)  # Zoom into the dense region

# Distribution of AveRooms
axes[1].hist(df['AveRooms'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[1].set_xlabel('Average Rooms per Dwelling')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of AveRooms')
axes[1].set_xlim(0, 15)

plt.tight_layout()
plt.show()

print("\nNote: There are outliers in AveRooms. We will handle them in preprocessing.")

---
## Cell 5 — Data Preprocessing

In [ ]:
# ---- Step 1: Select single feature and target ----
X = df[['AveRooms']].values   # Shape: (n, 1)
y = df['MedHouseVal'].values   # Shape: (n,)

print(f"Before outlier removal: {X.shape[0]} samples")

# ---- Step 2: Remove outliers (AveRooms > 15 are rare extreme values) ----
mask = X.flatten() <= 15
X = X[mask]
y = y[mask]
print(f"After outlier removal:  {X.shape[0]} samples")

# ---- Step 3: Train-Test Split (80/20) ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

# ---- Step 4: Feature Scaling (needed for Gradient Descent) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled feature mean: {X_train_scaled.mean():.6f} (≈ 0)")
print(f"Scaled feature std:  {X_train_scaled.std():.6f} (≈ 1)")
print("\nPreprocessing complete!")

---
## Cell 6 — Method 1: Linear Regression using Gradient Descent (From Scratch)

In [ ]:
class LinearRegressionGD:
    """
    Linear Regression using Batch Gradient Descent.
    
    Human Analogy:
    - Imagine learning to throw darts at a target.
    - Each throw (iteration), you see how far off you were (loss).
    - You adjust your aim slightly (update weights) in the direction
      that reduces the error.
    - Learning rate = how big each adjustment is.
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None       # Parameters [bias, weight]
        self.cost_history = []   # Track loss over iterations
    
    def fit(self, X, y):
        m = len(y)  # Number of training samples
        
        # Add bias column (column of 1s) to X
        X_b = np.c_[np.ones((m, 1)), X]  # Shape: (m, 2)
        
        # Initialize parameters to zeros
        self.theta = np.zeros(X_b.shape[1])  # [theta_0, theta_1]
        
        # Gradient Descent Loop
        for i in range(self.n_iterations):
            # 1. Predict: y_hat = X_b . theta
            y_pred = X_b.dot(self.theta)
            
            # 2. Compute error
            error = y_pred - y
            
            # 3. Compute gradients: (1/m) * X^T . error
            gradients = (1/m) * X_b.T.dot(error)
            
            # 4. Update parameters: theta = theta - lr * gradients
            self.theta -= self.learning_rate * gradients
            
            # 5. Track cost (MSE)
            cost = (1/(2*m)) * np.sum(error ** 2)
            self.cost_history.append(cost)
        
        return self
    
    def predict(self, X):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return X_b.dot(self.theta)


# ---- Train the model ----
gd_model = LinearRegressionGD(learning_rate=0.1, n_iterations=1000)
gd_model.fit(X_train_scaled, y_train)

print("=" * 50)
print("GRADIENT DESCENT RESULTS")
print("=" * 50)
print(f"Bias (theta_0):   {gd_model.theta[0]:.4f}")
print(f"Weight (theta_1): {gd_model.theta[1]:.4f}")
print(f"Final cost:       {gd_model.cost_history[-1]:.4f}")
print(f"Iterations:       {len(gd_model.cost_history)}")

---
## Cell 7 — Visualize Gradient Descent Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost over all iterations
axes[0].plot(gd_model.cost_history, color='crimson', linewidth=1.5)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost (MSE / 2)')
axes[0].set_title('Gradient Descent Convergence')

# Zoomed: first 100 iterations (where most learning happens)
axes[1].plot(gd_model.cost_history[:100], color='crimson', linewidth=1.5)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost (MSE / 2)')
axes[1].set_title('Convergence — First 100 Iterations (Zoomed)')

plt.tight_layout()
plt.show()

print("\nThe cost drops rapidly in early iterations, then plateaus — just like")
print("how humans learn fast initially and then improvements become marginal.")

---
## Cell 8 — Method 2: Linear Regression using the Normal Equation

In [ ]:
class LinearRegressionNormalEq:
    """
    Linear Regression using the Normal Equation (Closed-Form Solution).
    
    Formula: theta = (X^T X)^(-1) X^T y
    
    Human Analogy:
    - Instead of trial-and-error (gradient descent), this is like
      solving a math equation directly to get the exact answer.
    - Like using a formula to find the answer in one step vs
      guessing and improving repeatedly.
    
    Note: No learning rate, no iterations, no feature scaling needed!
    """
    
    def __init__(self):
        self.theta = None
    
    def fit(self, X, y):
        m = X.shape[0]
        # Add bias column
        X_b = np.c_[np.ones((m, 1)), X]
        
        # Normal Equation: theta = (X^T X)^(-1) X^T y
        self.theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        return self
    
    def predict(self, X):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return X_b.dot(self.theta)


# ---- Train (using UNSCALED data — normal equation doesn't need scaling) ----
ne_model = LinearRegressionNormalEq()
ne_model.fit(X_train, y_train)

print("=" * 50)
print("NORMAL EQUATION RESULTS")
print("=" * 50)
print(f"Bias (theta_0):   {ne_model.theta[0]:.4f}")
print(f"Weight (theta_1): {ne_model.theta[1]:.4f}")
print(f"\nInterpretation:")
print(f"  Base price ≈ ${ne_model.theta[0]*100000:,.0f}")
print(f"  Each additional room adds ≈ ${ne_model.theta[1]*100000:,.0f} to the price")

---
## Cell 9 — Method 3: Scikit-learn (Verification)

In [ ]:
# Scikit-learn's implementation (uses Normal Equation internally)
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

print("=" * 50)
print("SCIKIT-LEARN RESULTS (Verification)")
print("=" * 50)
print(f"Bias (intercept):  {sk_model.intercept_:.4f}")
print(f"Weight (coef):     {sk_model.coef_[0]:.4f}")

print(f"\n--- Comparison ---")
print(f"{'Method':<20} {'Bias':>10} {'Weight':>10}")
print("-" * 42)
print(f"{'Normal Equation':<20} {ne_model.theta[0]:>10.4f} {ne_model.theta[1]:>10.4f}")
print(f"{'Scikit-learn':<20} {sk_model.intercept_:>10.4f} {sk_model.coef_[0]:>10.4f}")
print(f"{'Gradient Descent*':<20} {gd_model.theta[0]:>10.4f} {gd_model.theta[1]:>10.4f}")
print("\n* GD values are on scaled features — so they differ in scale but")
print("  produce equivalent predictions when used with scaled inputs.")

---
## Cell 10 — Model Evaluation

In [ ]:
# Predictions on test set
y_pred_gd = gd_model.predict(X_test_scaled)    # GD uses scaled features
y_pred_ne = ne_model.predict(X_test)             # Normal Eq uses raw features
y_pred_sk = sk_model.predict(X_test)             # Sklearn uses raw features

def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'Method': name, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

results = [
    evaluate('Gradient Descent', y_test, y_pred_gd),
    evaluate('Normal Equation', y_test, y_pred_ne),
    evaluate('Scikit-learn', y_test, y_pred_sk),
]

results_df = pd.DataFrame(results)
print("=" * 60)
print("MODEL EVALUATION ON TEST SET")
print("=" * 60)
print(results_df.to_string(index=False))

print(f"\n--- Interpretation ---")
print(f"MSE:  Average squared error in predictions")
print(f"RMSE: Avg prediction is off by ~${results[1]['RMSE']*100000:,.0f}")
print(f"R²:   Model explains {results[1]['R²']*100:.1f}% of the variance")
print(f"\nNote: Low R² is expected — housing price depends on MANY factors,")
print(f"not just the number of rooms. This is a single-variable model.")

---
## Cell 11 — Visualization: Fitted Regression Line with Data Points

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ---- Plot 1: Gradient Descent (on scaled data) ----
axes[0].scatter(X_test_scaled, y_test, alpha=0.2, s=8, color='steelblue', label='Actual')
# Sort for a clean line
sort_idx = X_test_scaled.flatten().argsort()
axes[0].plot(X_test_scaled[sort_idx], y_pred_gd[sort_idx], color='crimson',
             linewidth=2, label='Fitted Line (GD)')
axes[0].set_xlabel('AveRooms (Scaled)')
axes[0].set_ylabel('Median House Value ($100K)')
axes[0].set_title('Gradient Descent')
axes[0].legend()

# ---- Plot 2: Normal Equation (on raw data) ----
axes[1].scatter(X_test, y_test, alpha=0.2, s=8, color='steelblue', label='Actual')
sort_idx = X_test.flatten().argsort()
axes[1].plot(X_test[sort_idx], y_pred_ne[sort_idx], color='darkorange',
             linewidth=2, label='Fitted Line (NE)')
axes[1].set_xlabel('AveRooms')
axes[1].set_ylabel('Median House Value ($100K)')
axes[1].set_title('Normal Equation')
axes[1].legend()

# ---- Plot 3: Scikit-learn (on raw data) ----
axes[2].scatter(X_test, y_test, alpha=0.2, s=8, color='steelblue', label='Actual')
axes[2].plot(X_test[sort_idx], y_pred_sk[sort_idx], color='green',
             linewidth=2, label='Fitted Line (sklearn)')
axes[2].set_xlabel('AveRooms')
axes[2].set_ylabel('Median House Value ($100K)')
axes[2].set_title('Scikit-learn')
axes[2].legend()

plt.suptitle('Linear Regression — Fitted Line vs Actual Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Cell 12 — Combined Visualization (Best Plot for Report)

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot of test data
plt.scatter(X_test, y_test, alpha=0.15, s=10, color='steelblue', label='Test Data')

# Fitted line (Normal Equation = sklearn, so just one line needed)
sort_idx = X_test.flatten().argsort()
plt.plot(X_test[sort_idx], y_pred_ne[sort_idx], color='crimson',
         linewidth=2.5, label=f'Fitted Line (y = {ne_model.theta[1]:.3f}x + {ne_model.theta[0]:.3f})')

# Annotations
plt.xlabel('Average Rooms per Dwelling (AveRooms)', fontsize=12)
plt.ylabel('Median House Value ($100,000s)', fontsize=12)
plt.title('Simple Linear Regression — California Housing Dataset', fontsize=14)
plt.legend(fontsize=11)

# Add metrics text box
metrics_text = f"MSE  = {results[1]['MSE']:.4f}\nRMSE = {results[1]['RMSE']:.4f}\nR²   = {results[1]['R²']:.4f}"
plt.text(0.02, 0.97, metrics_text, transform=plt.gca().transAxes,
         fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---
## Cell 13 — Residual Analysis

In [ ]:
residuals = y_test - y_pred_ne

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_ne, residuals, alpha=0.15, s=8, color='steelblue')
axes[0].axhline(y=0, color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values')

# Residual distribution
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[1].axvline(x=0, color='crimson', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

print("Observations:")
print("- Non-random residual pattern suggests a linear model is too simple.")
print("- This is expected — housing prices have complex, non-linear relationships.")
print("- A single feature cannot capture all the variance in house prices.")

---
## Cell 14 — Effect of Learning Rate on Gradient Descent

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

plt.figure(figsize=(10, 6))

for lr, color in zip(learning_rates, colors):
    model = LinearRegressionGD(learning_rate=lr, n_iterations=200)
    model.fit(X_train_scaled, y_train)
    plt.plot(model.cost_history, label=f'lr = {lr}', color=color, linewidth=2)

plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Effect of Learning Rate on Gradient Descent Convergence')
plt.legend(fontsize=11)
plt.ylim(0, 3)  # Cap y-axis for readability
plt.tight_layout()
plt.show()

print("Human Analogy:")
print("- Too small lr (0.001): Like taking tiny baby steps — you'll get there, but slowly.")
print("- Just right lr (0.1):  Confident strides — efficient learning.")
print("- Too large lr (0.5):   Jumping wildly — might overshoot or never settle down.")

---
## Cell 15 — Summary & Key Takeaways

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║              SUMMARY & KEY TAKEAWAYS                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Dataset: California Housing (20,640 samples)                ║
║  Feature: AveRooms (average rooms per dwelling)              ║
║  Target:  Median House Value (in $100,000s)                  ║
║                                                              ║
║  Methods Implemented:                                        ║
║  1. Gradient Descent — iterative, needs scaling & lr tuning  ║
║  2. Normal Equation  — exact, one-step, no hyperparameters   ║
║  3. Scikit-learn     — production-ready, matches Normal Eq   ║
║                                                              ║
║  Results:                                                    ║
║  • All three methods converge to the same solution ✓         ║
║  • Low R² is expected with a single feature                  ║
║  • Adding more features (multivariate) would improve R²      ║
║                                                              ║
║  Human Learning Parallels:                                   ║
║  • Gradient Descent = Trial & error with feedback             ║
║  • Normal Equation  = Solving with a formula directly         ║
║  • Overfitting       = Memorizing vs understanding            ║
║  • Learning Rate     = Size of each adjustment step           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")